# 11. Final End-to-End Prediction Demonstration
**Project:** Academic Sports Machine Learning Project  
**Domain:** Cricket Player Performance Analytics  
**Course Context:** Module 1, 2 & 3 (ML System Deployment, Inference Pipeline & Simulation)  

---

## 1. Objectives & Academic Demonstration
The final step of the Machine Learning lifecycle is serving actionable predictions from raw, unseen inputs.
In this notebook, we demonstrate:
1. Loading the persisted preprocessing pipeline (`preprocessor.joblib`) and champion model (`champion_pipeline.joblib`).
2. **Interactive Single-Player Prediction Simulation**:
   - Evaluating three distinct profiles: (a) Elite Top-Order Batsman, (b) Strike Bowler, (c) Struggling All-Rounder.
   - Returning class prediction, calibrated probability score, and domain interpretation.
3. **Batch Match-Day Simulation**:
   - Simulating a full 22-player match roster between two teams (Team A vs Team B).
   - Selecting the **Match-Day Elite XI** based on top predicted probability scores.
4. Exporting final predictions to `outputs/predictions/final_predictions.csv`.
5. Final viva defense summary and project conclusions.


In [5]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from pathlib import Path

from src.feature_engineering import engineer_cricket_features
from src.utils import MODELS_DIR, PREDICTIONS_DIR, TABLES_DIR, load_artifact, set_seed

set_seed(42)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

# Load pipeline artifacts
preprocessor = load_artifact(MODELS_DIR / "preprocessor.joblib")
champion_model = load_artifact(MODELS_DIR / "champion_pipeline.joblib")
print("Pipeline and Champion Model loaded successfully!")

Pipeline and Champion Model loaded successfully!


## 2. End-to-End Inference Function
This function takes arbitrary raw player records, engineers domain features, applies preprocessing, and outputs predictions and confidence scores.

In [6]:
def predict_player_elite_status(raw_players_df, threshold=0.44):
    """
    End-to-End inference pipeline:
    1. Engineers cricket domain metrics
    2. Transforms features via fitted preprocessor
    3. Generates class probability and binary prediction
    """
    df_eng = engineer_cricket_features(raw_players_df)
    X_proc = preprocessor.transform(df_eng)
    
    probas = champion_model.predict_proba(X_proc)[:, 1]
    preds = (probas >= threshold).astype(int)
    
    results = raw_players_df[['Player_ID', 'Player_Name', 'Role', 'Team', 'Opponent', 'Match_Format']].copy()
    results['Predicted_Probability'] = probas.round(4)
    results['Elite_Performer_Status'] = np.where(preds == 1, 'YES (Elite Performer)', 'NO (Standard)')
    results['Recommendation'] = np.where(
        probas >= 0.70, 'Must-Select Star / Player of the Match Contender',
        np.where(probas >= 0.44, 'Contender / Squad Core Selection', 'Standard Rotation / Development Player')
    )
    return results

## 3. Simulation 1: Single Player Interactive Profiles
Simulating three distinct real-world cricket scenarios.

In [7]:
sample_profiles = pd.DataFrame([
    {
        'Player_ID': 'SIM_001',
        'Player_Name': 'Virat Sterling (Star Batsman)',
        'Age': 32,
        'Role': 'Batsman',
        'Team': 'Team_A',
        'Match_ID': 'SIM_M01',
        'Match_Date': '2026-10-15',
        'Match_Format': 'T20',
        'Opponent': 'Team_B',
        'Venue': 'Stadium_01',
        'Home_Away': 'Home',
        'Experience_Years': 12,
        'Matches_Played': 140,
        'Runs': 88,
        'Balls_Faced': 52,
        'Fours': 8,
        'Sixes': 4,
        'Strike_Rate': 169.23,
        'Batting_Average': 58.5,
        'Wickets': 0,
        'Overs_Bowled': 0.0,
        'Runs_Conceded': 0.0,
        'Economy': 0.0,
        'Catches': 2,
        'Stumpings': 0,
        'Recent_Form': 85.0,
        'Previous_Performance': 80.0,
        'Fitness_Score': 92.0,
        'Consistency_Score': 88.0,
        'Pressure_Score': 75.0
    },
    {
        'Player_ID': 'SIM_002',
        'Player_Name': 'Jasprit Bolt (Strike Bowler)',
        'Age': 28,
        'Role': 'Bowler',
        'Team': 'Team_A',
        'Match_ID': 'SIM_M01',
        'Match_Date': '2026-10-15',
        'Match_Format': 'T20',
        'Opponent': 'Team_B',
        'Venue': 'Stadium_01',
        'Home_Away': 'Home',
        'Experience_Years': 8,
        'Matches_Played': 95,
        'Runs': 10,
        'Balls_Faced': 6,
        'Fours': 1,
        'Sixes': 0,
        'Strike_Rate': 166.6,
        'Batting_Average': 18.0,
        'Wickets': 4,
        'Overs_Bowled': 4.0,
        'Runs_Conceded': 18.0,
        'Economy': 4.5,
        'Catches': 1,
        'Stumpings': 0,
        'Recent_Form': 82.0,
        'Previous_Performance': 78.0,
        'Fitness_Score': 89.0,
        'Consistency_Score': 85.0,
        'Pressure_Score': 80.0
    },
    {
        'Player_ID': 'SIM_003',
        'Player_Name': 'Rookie Davis (Bench Player)',
        'Age': 20,
        'Role': 'All-Rounder',
        'Team': 'Team_B',
        'Match_ID': 'SIM_M01',
        'Match_Date': '2026-10-15',
        'Match_Format': 'T20',
        'Opponent': 'Team_A',
        'Venue': 'Stadium_01',
        'Home_Away': 'Away',
        'Experience_Years': 1,
        'Matches_Played': 12,
        'Runs': 14,
        'Balls_Faced': 20,
        'Fours': 1,
        'Sixes': 0,
        'Strike_Rate': 70.0,
        'Batting_Average': 22.0,
        'Wickets': 0,
        'Overs_Bowled': 2.0,
        'Runs_Conceded': 24.0,
        'Economy': 12.0,
        'Catches': 0,
        'Stumpings': 0,
        'Recent_Form': 35.0,
        'Previous_Performance': 40.0,
        'Fitness_Score': 70.0,
        'Consistency_Score': 45.0,
        'Pressure_Score': 60.0
    }
])

sim_results = predict_player_elite_status(sample_profiles)
print("=== Individual Profile Simulation Results ===")
print(sim_results[['Player_Name', 'Role', 'Predicted_Probability', 'Elite_Performer_Status', 'Recommendation']].to_string(index=False))

=== Individual Profile Simulation Results ===
                  Player_Name        Role  Predicted_Probability Elite_Performer_Status                                   Recommendation
Virat Sterling (Star Batsman)     Batsman                 0.9989  YES (Elite Performer) Must-Select Star / Player of the Match Contender
 Jasprit Bolt (Strike Bowler)      Bowler                 0.9931  YES (Elite Performer) Must-Select Star / Player of the Match Contender
  Rookie Davis (Bench Player) All-Rounder                 0.0000          NO (Standard)           Standard Rotation / Development Player


## 4. Simulation 2: Batch Match-Day Roster Simulation (22 Players)
Simulating 22 players from the test set to choose the Match-Day Elite XI.

In [8]:
# Load raw test set data
from src.preprocessing import load_raw_data
df_full = load_raw_data()
match_sample = df_full.iloc[100:122].copy()

match_preds = predict_player_elite_status(match_sample)
match_preds_sorted = match_preds.sort_values('Predicted_Probability', ascending=False).reset_index(drop=True)

print("=== Match-Day Roster Predictions (Top 11 Selected as Dream XI) ===")
print(match_preds_sorted[['Player_Name', 'Role', 'Team', 'Predicted_Probability', 'Elite_Performer_Status']].head(11).to_string())

# Save predictions
match_preds_sorted.to_csv(PREDICTIONS_DIR / "final_predictions.csv", index=False)
print(f"[SUCCESS] Predictions exported to {PREDICTIONS_DIR / 'final_predictions.csv'}")

=== Match-Day Roster Predictions (Top 11 Selected as Dream XI) ===
     Player_Name           Role    Team  Predicted_Probability Elite_Performer_Status
0   Player_00119         Bowler  Team_E                 0.9908  YES (Elite Performer)
1   Player_00107         Bowler  Team_I                 0.9876  YES (Elite Performer)
2   Player_00114         Bowler  Team_A                 0.9628  YES (Elite Performer)
3   Player_00105    All-Rounder  Team_D                 0.9484  YES (Elite Performer)
4   Player_00113        Batsman  Team_G                 0.4100          NO (Standard)
5   Player_00110  Wicket-Keeper  Team_I                 0.3472          NO (Standard)
6   Player_00104  Wicket-Keeper  Team_A                 0.2623          NO (Standard)
7   Player_00106    All-Rounder  Team_G                 0.1641          NO (Standard)
8   Player_00122         Bowler  Team_I                 0.1124          NO (Standard)
9   Player_00103        Batsman  Team_G                 0.0744          N

## 5. Summary of Project Achievements & Viva Defense Checklist

### Academic Project Summary
1. **Phase 0 & 1**: Clean inspection of 10,000 cricket match records; formulation of the **Elite Performance Classification Task** (`High_Performer = 1` for `Performance_Score >= 60.0`), yielding an academically rigorous **~19.04% imbalanced problem**.
2. **Phase 2 & 3**: Zero-leakage data preparation; stratified 80/20 train/test split; domain-specific cricket feature engineering (`Batting_Impact`, `Bowling_Impact`, `Boundary_Run_Ratio`).
3. **Phase 4**: Baseline establishment exposed the **Accuracy Paradox** (80.96% dummy accuracy with 0.0 recall).
4. **Phase 5**: Systematic imbalance experimentation proved that **Cost-Sensitive Learning** and **SMOTE** elevate minority recall from 0.60 to >0.87.
5. **Phase 6 & 7**: Advanced ensemble modeling and Bayesian/Randomized search revealed **Tuned LightGBM** as the champion architecture (ROC-AUC ~0.917, PR-AUC ~0.751, F1 ~0.674).
6. **Phase 8**: Interpretability analysis via **SHAP** provided game-theoretic verification of sports domain intuitions.
7. **Phase 9**: Production inference pipeline demonstrated end-to-end reliability on both individual and batch match-day simulations.

### Viva Defense Checklist
- [x] **Leakage Prevention**: Split before scaling; identifiers removed; raw target dropped from $X$.
- [x] **Imbalance Rigor**: SMOTE applied only within CV folds; evaluated via PR-AUC and Balanced Accuracy.
- [x] **Model Breadth**: Evaluated linear, tree, bagging (RF, ExtraTrees), and boosting (XGBoost, LightGBM).
- [x] **Interpretability**: Quantified global and local feature impacts with SHAP TreeExplainer.
- [x] **Operational Value**: Tuned decision threshold $\tau^* = 0.44$ to align with sports selection costs.
